In [5]:
import os
import shutil
from skimage import measure
import numpy as np
import geopandas as gpd
from osgeo import gdal, ogr
import rasterio
import sys

origin = '/workspace/'
sys.path.append('/media/')

from FieldWaterUseTools.FuncBox.Misc import assert_same_length, getFilelist, path_safe, makeTif_np_to_matching_tif, getSpatRefRas, getSpatRefVec

year = 2023
states = ['Brandenburg', 'Brandenburg']
models = ['FromScratch_dilate_T', 'FromScratch_dilate_T']
t_exts = ['03', '03']
t_bounds = ['01', '01']
maskVersions = ['ThuenenMasked', 'UnMasked']

idx = 0
state = states[idx]
para_id = f"ext_{t_exts[idx]}_bound_{t_bounds[idx]}"
outPath = f"{origin}fields/07_Polygonized/{state}/{models[idx]}/{year}/{maskVersions[idx]}/"


# load vector ds to be simplified
vector_path = f"{outPath}{maskVersions[idx]}_{para_id}.gpkg"
in_ds = ogr.Open(vector_path)
layer = in_ds.GetLayer(0)  # or "polygons"?

In [3]:
SUBSET = 500        # set to None to process all features
BUFFER_DIST = 5     # half pixel (10m res → 5m)
SIMPLIFY_TOL = 5    # metres; tune between 5–15

# ── output layer ────────────────────────────────────────
outPath = f"{vector_path.split('.')[0]}_smooth_test_buff_{BUFFER_DIST}_simplTol_{SIMPLIFY_TOL}.gpkg"
driver = ogr.GetDriverByName('GPKG')
out_ds = driver.CreateDataSource(outPath)
out_layer = out_ds.CreateLayer('polygons', getSpatRefVec(in_ds), geom_type=ogr.wkbPolygon)

layer_defn = layer.GetLayerDefn()
for i in range(layer_defn.GetFieldCount()):
    out_layer.CreateField(layer_defn.GetFieldDefn(i))

layer.ResetReading()
skipped = 0

for idx1, feature in enumerate(layer):
    if SUBSET is not None and idx1 >= SUBSET:
        break
    if idx1 % 100 == 0:
        print(f"{idx1} / {SUBSET or 'all'}")

    geom = feature.GetGeometryRef()
    if geom is None:
        skipped += 1
        continue

    # buffer close → removes staircase notches, then simplify
    simplified = (geom
        .Buffer(BUFFER_DIST)
        .Buffer(-BUFFER_DIST)
        .SimplifyPreserveTopology(SIMPLIFY_TOL))

    if simplified is None or simplified.IsEmpty():
        skipped += 1
        continue

    new_feature = ogr.Feature(out_layer.GetLayerDefn())
    new_feature.SetGeometry(simplified)
    for i in range(feature.GetFieldCount()):
        val = feature.GetField(i)
        if val is not None:
            new_feature.SetField(i, val)
    out_layer.CreateFeature(new_feature)

print(f"Done. Skipped {skipped} null/empty geometries.")
out_layer = None
out_ds = None

0 / 500
100 / 500
200 / 500
300 / 500
400 / 500
Done. Skipped 0 null/empty geometries.


In [9]:
SUBSET = 500        # set to None to process all features
BUFFER_DIST = 13    # half pixel (10m res → 5m)

# ── output layer ────────────────────────────────────────
outPath = f"{vector_path.split('.')[0]}_smooth_test_buff_{BUFFER_DIST}_agro.gpkg"
driver = ogr.GetDriverByName('GPKG')
out_ds = driver.CreateDataSource(outPath)
out_layer = out_ds.CreateLayer('polygons', getSpatRefVec(in_ds), geom_type=ogr.wkbPolygon)

layer_defn = layer.GetLayerDefn()
for i in range(layer_defn.GetFieldCount()):
    out_layer.CreateField(layer_defn.GetFieldDefn(i))

layer.ResetReading()
skipped = 0

for idx1, feature in enumerate(layer):
    if SUBSET is not None and idx1 >= SUBSET:
        break
    if idx1 % 100 == 0:
        print(f"{idx1} / {SUBSET or 'all'}")

    geom = feature.GetGeometryRef()
    if geom is None:
        skipped += 1
        continue

    # buffer close → removes staircase notches, then simplify
    simplified = (geom
        .Buffer(BUFFER_DIST)
        .Buffer(-BUFFER_DIST)
        .SimplifyPreserveTopology(15))

    new_feature = ogr.Feature(out_layer.GetLayerDefn())
    new_feature.SetGeometry(simplified)
    for i in range(feature.GetFieldCount()):
        val = feature.GetField(i)
        if val is not None:
            new_feature.SetField(i, val)
    out_layer.CreateFeature(new_feature)

out_layer = None
out_ds = None

0 / 500
100 / 500
200 / 500
300 / 500
400 / 500


In [16]:
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
import numpy as np

# ── Chaikin smoothing (mirrors QGIS "smooth" algorithm) ─
def chaikin_smooth_ring(coords, iterations=3):
    """
    Densify by midpoints then cut corners — equivalent to:
    'densify by count (1 vertex)' → 'extract vertices' →
    'keep only even-index vertices' → 'points to path' → 'smooth'
    """
    pts = np.array(coords)
    # ensure ring is closed
    if not np.allclose(pts[0], pts[-1]):
        pts = np.vstack([pts, pts[0]])

    for _ in range(iterations):
        new_pts = []
        for i in range(len(pts) - 1):
            p0, p1 = pts[i], pts[i + 1]
            # Q: 25% along segment, R: 75% along segment
            Q = p0 + SMOOTH_ALPHA       * (p1 - p0)
            R = p0 + (1 - SMOOTH_ALPHA) * (p1 - p0)
            new_pts.extend([Q, R])
        # close the ring
        new_pts.append(new_pts[0])
        pts = np.array(new_pts)

    return pts.tolist()

def smooth_polygon(geom, iterations=3):
    """Apply Chaikin smoothing to all rings of a Polygon or MultiPolygon."""
    def smooth_poly(poly):
        exterior = chaikin_smooth_ring(poly.exterior.coords, iterations)
        interiors = [
            chaikin_smooth_ring(ring.coords, iterations)
            for ring in poly.interiors
        ]
        return Polygon(exterior, interiors)

    if geom.geom_type == 'Polygon':
        return smooth_poly(geom)
    elif geom.geom_type == 'MultiPolygon':
        return MultiPolygon([smooth_poly(p) for p in geom.geoms])
    else:
        return geom  # pass through non-polygon types




SUBSET       = 500
SMOOTH_ITER  = 2
SMOOTH_ALPHA = 0.25   # Chaikin tension: 0.25 is standard, increase for more smoothing
INPUT_PATH   = vector_path
OUTPUT_PATH  = f"{vector_path.split('.')[0]}_chaikin_smooth_iter_{SMOOTH_ITER}_alph_{SMOOTH_ALPHA}.gpkg"

alphas = [r/100 for r in range(15,85,5)]

for SMOOTH_ALPHA in alphas:
    gdf = gpd.read_file(INPUT_PATH)

    if SUBSET is not None:
        gdf = gdf.iloc[:SUBSET].copy()
        print(f"Testing on {SUBSET} features")

    print("Smoothing geometries...")
    gdf['geometry'] = gdf['geometry'].apply(
        lambda g: smooth_polygon(g, iterations=SMOOTH_ITER)
    )

    # drop any geometries that collapsed
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]

    print(f"Writing {len(gdf)} features → {OUTPUT_PATH}")
    gdf.to_file(OUTPUT_PATH, driver='GPKG')
    print("Done.")

Testing on 500 features
Smoothing geometries...
Writing 500 features → /workspace/fields/07_Polygonized/Brandenburg/FromScratch_dilate_T/2023/ThuenenMasked/ThuenenMasked_ext_03_bound_01_chaikin_smooth_iter_2_alph_0.25.gpkg
Done.
Testing on 500 features
Smoothing geometries...
Writing 500 features → /workspace/fields/07_Polygonized/Brandenburg/FromScratch_dilate_T/2023/ThuenenMasked/ThuenenMasked_ext_03_bound_01_chaikin_smooth_iter_2_alph_0.25.gpkg
Done.
Testing on 500 features
Smoothing geometries...
Writing 500 features → /workspace/fields/07_Polygonized/Brandenburg/FromScratch_dilate_T/2023/ThuenenMasked/ThuenenMasked_ext_03_bound_01_chaikin_smooth_iter_2_alph_0.25.gpkg
Done.
Testing on 500 features
Smoothing geometries...
Writing 500 features → /workspace/fields/07_Polygonized/Brandenburg/FromScratch_dilate_T/2023/ThuenenMasked/ThuenenMasked_ext_03_bound_01_chaikin_smooth_iter_2_alph_0.25.gpkg
Done.
Testing on 500 features
Smoothing geometries...
Writing 500 features → /workspace/fi